In [1]:
import os
import pandas as pd
import ast
from itertools import chain
from collections import Counter
import pickle
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from tqdm.notebook import tqdm
tqdm.pandas()

In [ ]:
db_file_path = '/mnt/share/kaichixie/A2H_preclinical_database/outputs/preclinical_db/v2/a2h_database_version_2.xlsx'
pcdb = pd.read_excel(db_file_path, engine="openpyxl", dtype=str,
                       sheet_name='A2H_database_version_2', keep_default_na=False)

unmapped_extractions = pd.read_excel(db_file_path,
                          sheet_name="unmapped_extractions", engine="openpyxl", dtype=str,
                          keep_default_na=False)

with open("/mnt/share/kaichixie/A2H_preclinical_database/data/mesh/mesh_synonyms_look_up_table.pkl", 'rb') as f:
    mesh_synonyms_LUT = pickle.load(f)
    
with open("/mnt/share/kaichixie/A2H_preclinical_database/data/drugbank/drugbank_synonyms_look_up_table.pkl", 'rb') as f:
    drugbank_synonyms_LUT = pickle.load(f)

with open("/mnt/share/kaichixie/A2H_preclinical_database/data/drugbank/drugbank_common_name_look_up_table.pkl", 'rb') as f:
    drugbank_common_name_LUT = pickle.load(f)

with open("/mnt/share/kaichixie/A2H_preclinical_database/data/UMLS/UMLS_synonyms_look_up_table.pkl", 'rb') as f:
    umls_synonyms_LUT = pickle.load(f)

pcdb["drug_external_ids"] = pcdb["drug_external_ids"].apply(lambda x: ast.literal_eval(x) if x != '' else x)

In [142]:
pcdb_drug = pcdb.explode("drug_external_ids")
pcdb_drug = pcdb_drug[pcdb_drug['drug_external_ids']!=""].copy()

In [143]:
pcdb_drug["drugbank_id"] = \
    pcdb_drug["drug_external_ids"].apply(lambda x : x['drugbank'] if ('drugbank' in x) else "")
pcdb_drug = pcdb_drug[pcdb_drug['drugbank_id']!='']

drugbank_ids = pcdb_drug['drugbank_id'].to_list()
drugbank_ids = list(chain.from_iterable(drugbank_ids))
drugbank_ids = list(set(drugbank_ids))

In [144]:
pcdb_drug_entities = \
    pd.DataFrame.from_dict({"drugbank_id": drugbank_ids})

drug_PCDB_ids = [f"PCDB_DR{i}" for i in range(1, len(pcdb_drug_entities)+1)]
pcdb_drug_entities["PCDB_id"] = drug_PCDB_ids

In [145]:
drugbank_ids_to_mesh_ids = {}
for db_id in pcdb_drug_entities['drugbank_id'].unique():
    drugbank_ids_to_mesh_ids[db_id] = set()

for d_eids in tqdm(pcdb["drug_external_ids"].to_list(), total=len(pcdb)):
    for d_eid in d_eids:
        if 'drugbank' in d_eid and 'mesh' in d_eid:
            db_ids = d_eid['drugbank']
            for db_id in db_ids:
                if db_id in drugbank_ids_to_mesh_ids:
                    for mesh_id in d_eid['mesh']:
                        drugbank_ids_to_mesh_ids[db_id].add(mesh_id)

  0%|          | 0/186627 [00:00<?, ?it/s]

In [146]:
pcdb_drug_entities["mesh_id"] = \
    pcdb_drug_entities["drugbank_id"].apply(lambda x: list(drugbank_ids_to_mesh_ids[x]) if len(drugbank_ids_to_mesh_ids[x]) != 0 else "")

In [147]:
mesh_ids_linked_drugbank = \
    list(chain.from_iterable(pcdb_drug_entities['mesh_id'].to_list()))
mesh_ids_linked_drugbank = set(mesh_ids_linked_drugbank)

In [148]:
pcdb_drug_MeSH = pcdb.explode("drug_external_ids")
pcdb_drug_MeSH = pcdb_drug_MeSH[pcdb_drug_MeSH['drug_external_ids']!=""]

pcdb_drug_MeSH["drug_mesh_id"] = \
    pcdb_drug_MeSH["drug_external_ids"].apply(lambda x : x['mesh'] if ('mesh' in x) else "")
pcdb_drug_MeSH = pcdb_drug_MeSH[pcdb_drug_MeSH['drug_mesh_id']!='']

pcdb_drug_MeSH = pcdb_drug_MeSH.explode("drug_mesh_id")
all_mapped_drug_mesh_ids = pcdb_drug_MeSH['drug_mesh_id'].unique().tolist()
mapped_drug_mesh_ids_not_linked_drugbank = \
    [x for x in all_mapped_drug_mesh_ids if x not in mesh_ids_linked_drugbank]

In [149]:
mesh_data = pd.read_csv('/mnt/share/kaichixie/A2H_preclinical_database/data/mesh/mesh_data.tsv',
                        sep='\t')
mesh_data = mesh_data.set_index("mesh_id")

In [150]:
def create_tree_id_lut():
    mesh_desc_data = \
        pd.read_csv("/mnt/share/kaichixie/A2H_preclinical_database/data/mesh/mesh_descriptive_data.tsv",
                    sep="\t")
    mesh_desc_data["tree_numbers"] = \
        mesh_desc_data["tree_numbers"].apply(lambda x: ast.literal_eval(x))
    tree_id_lut = {}
    
    for _, row in mesh_desc_data.iterrows():
        # print(row['mesh_id'])
        tree_id_lut[row['mesh_id']] = row['tree_numbers']
    return tree_id_lut

mesh_tree_id_lut = create_tree_id_lut()

In [151]:
drug_mesh_ids_with_PA = set()
for m_id in mapped_drug_mesh_ids_not_linked_drugbank:
    if m_id not in mesh_tree_id_lut:
        pas = mesh_data.loc[m_id]['pharmacological_actions']
        pas = ast.literal_eval(pas)
        if len(pas) > 0:
            drug_mesh_ids_with_PA.add(m_id)
    else:
        tree_ids = mesh_tree_id_lut[m_id]
        top_tree_id = set([x.split(".")[0] for x in tree_ids])
        # print(top_tree_id)
        for tid in top_tree_id:
            if tid.startswith('D'):
                drug_mesh_ids_with_PA.add(m_id)
                break

In [152]:
current_id_max = len(pcdb_drug_entities)
for mesh_id in drug_mesh_ids_with_PA:
    new_row = pd.DataFrame.from_dict({
        "drugbank_id" : "",
        "PCDB_id": f"PCDB_DR{current_id_max+1}",
        "mesh_id": [mesh_id]
    })
    current_id_max = current_id_max + 1
    pcdb_drug_entities = pd.concat([pcdb_drug_entities, new_row],
                                   ignore_index=True)
    

In [153]:
drugbank_id_to_UMLS_id = {}
for db_id in pcdb_drug_entities[pcdb_drug_entities['drugbank_id']!='']['drugbank_id'].unique():
    drugbank_id_to_UMLS_id[db_id] = set()

mesh_id_to_UMLS_id = {}
all_mesh_id = pcdb_drug_entities.explode('mesh_id')
for m_id in all_mesh_id[all_mesh_id['mesh_id']!='']['mesh_id'].unique():
    mesh_id_to_UMLS_id[m_id] = set()

In [154]:
for d_eids in tqdm(pcdb["drug_external_ids"].to_list(), total=len(pcdb)):
    for d_eid in d_eids: # each drug mapping
        if 'drugbank' in d_eid and 'UMLS' in d_eid:
            db_ids = d_eid['drugbank']
            for db_id in db_ids:
                if db_id in drugbank_id_to_UMLS_id:
                    for umls_id in d_eid['UMLS']:
                        drugbank_id_to_UMLS_id[db_id].add(umls_id)
        if 'mesh' in d_eid and 'UMLS' in d_eid:
            mesh_ids = d_eid['mesh']
            for mesh_id in mesh_ids:
                if mesh_id in mesh_id_to_UMLS_id:
                    for umls_id in d_eid['UMLS']:
                        mesh_id_to_UMLS_id[mesh_id].add(umls_id)

  0%|          | 0/186627 [00:00<?, ?it/s]

In [155]:
# Update drug entities df
pcdb_drug_entities["UMLS_id"] = \
    pcdb_drug_entities["drugbank_id"].apply(lambda x: drugbank_id_to_UMLS_id[x] if x != ''  else '')

In [156]:
def find_UMLS_id_by_mesh_id(row):
    mesh_ids = row["mesh_id"]
    if mesh_ids == '':
        return ""
    if isinstance(mesh_ids, str):
        mesh_ids = [mesh_ids]
        
    curr_umls_ids = row["UMLS_id"]
    if curr_umls_ids == '':
        curr_umls_ids = set()
    for mesh_id in mesh_ids:
        if mesh_id in mesh_id_to_UMLS_id:
            curr_umls_ids.update(mesh_id_to_UMLS_id[mesh_id])
    if len(curr_umls_ids) == 0:
        return ''
    return curr_umls_ids
            
pcdb_drug_entities["UMLS_id"] = pcdb_drug_entities.apply(find_UMLS_id_by_mesh_id, axis=1)

In [157]:
pcdb_drug_entities["UMLS_id"] = \
    pcdb_drug_entities["UMLS_id"].apply(lambda x: list(x) if x != '' else x)

In [158]:
resolved_UMLS_ids = pcdb_drug_entities[pcdb_drug_entities["UMLS_id"]!='']["UMLS_id"].to_list()
resolved_UMLS_ids = list(chain.from_iterable(resolved_UMLS_ids))
resolved_UMLS_ids = set(resolved_UMLS_ids)

In [ ]:
unresolved_UMLS_ids = set()
for d_eids in tqdm(pcdb["drug_external_ids"].to_list(), total=len(pcdb)):
    for d_eid in d_eids: # each drug mapping
        if 'UMLS' in d_eid: # Only mapped to UMLS
            for umls_id in d_eid['UMLS']:
                if umls_id not in resolved_UMLS_ids:
                    # print(umls_id, " resolved")
                    unresolved_UMLS_ids.add(umls_id)

In [160]:
with open('/mnt/share/kaichixie/A2H_preclinical_database/data/UMLS/UMLS_semantic_LUT.pkl', 'rb') as f:
    umls_semantic_lut = pickle.load(f)

In [161]:
allowed_semantic_types = ['T195', #Antibiotic,
                          'T200', # Clinical drugs
                          'T121'] # Pharmalogical substance
unresolved_drug_UMLS_ids = set()
for umls_id in unresolved_UMLS_ids:
    s_types = umls_semantic_lut[umls_id]
    for s_t in s_types:
        if s_t in allowed_semantic_types:
            unresolved_drug_UMLS_ids.add(umls_id)

In [162]:
# 
current_id_max = len(pcdb_drug_entities)
for new_umls_id in unresolved_drug_UMLS_ids:
    new_row = pd.DataFrame.from_dict({
        "drugbank_id" : "",
        "PCDB_id": f"PCDB_DR{current_id_max+1}",
        "mesh_id": "",
        "UMLS_id": [new_umls_id]
    })
    current_id_max = current_id_max + 1
    pcdb_drug_entities = pd.concat([pcdb_drug_entities, new_row],
                                   ignore_index=True)
pcdb_drug_entities

,drugbank_id,PCDB_id,mesh_id,UMLS_id
0,DB01119,PCDB_DR1,[D003981],[C0012022]
1,DB12380,PCDB_DR2,[C579046],[C2981796]
2,DB01620,PCDB_DR3,[D010632],"[C0031408, C0887016]"
3,DB12064,PCDB_DR4,[C550356],[C2346911]
4,DB00440,PCDB_DR5,[D014295],"[C0041041, C0801990]"
...,...,...,...,...
11748,,PCDB_DR11749,,C1615056
11749,,PCDB_DR11750,,C0876102
11750,,PCDB_DR11751,,C0971163
11751,,PCDB_DR11752,,C4053715


In [187]:
# get drug name, and synonyms
def get_drug_synonyms(row):
    drugbank_id = row["drugbank_id"]
    mesh_id = row["mesh_id"]
    umls_id = row['UMLS_id']
    official_drug_name = ""
    official_synonyms = []
    if drugbank_id != "":
        official_drug_name = drugbank_common_name_LUT[drugbank_id]
    elif mesh_id != "":
        official_drug_name = mesh_synonyms_LUT[mesh_id][0]
    else:
        official_drug_name = f"UMLS_{umls_id}"
    
    if drugbank_id != "":
        official_synonyms = official_synonyms + drugbank_synonyms_LUT[drugbank_id]
    if mesh_id != "":
        if isinstance(mesh_id, str):
            mesh_id = [mesh_id]
        for m_id in mesh_id:
            official_synonyms = official_synonyms + mesh_synonyms_LUT[m_id]
    if umls_id != "":
        if isinstance(umls_id, str):
            umls_id = [umls_id]
        for u_id in umls_id:
            official_synonyms = official_synonyms + umls_synonyms_LUT[u_id]
    
    official_synonyms = [s.lower() for s in official_synonyms]
    official_synonyms = list(set(official_synonyms))
    return official_drug_name, official_synonyms
    

In [188]:
pcdb_drug_entities[["drug_name", "drug_synonyms"]] = \
    pcdb_drug_entities.apply(get_drug_synonyms, axis=1, result_type='expand')

In [195]:
def representative_name_for_cui(
    mrconso,
    cui,
    tty_priority=("PT", "PN", "MH", "HT", "SY"),
    sab_priority= None,
):

    # 1) filter rows for the CUI
    df = mrconso[mrconso["CUI"] == cui]
    # print(cui)
        

    # 3) scoring helpers (higher is better)
    tty_rank = {t: (len(tty_priority) - i) for i, t in enumerate(tty_priority)}
    sab_rank = {s: (len(sab_priority) - i) for i, s in enumerate(sab_priority)} if sab_priority else {}

    def score_row(r) -> tuple:
        # prefer ISPREF='Y', then TS='P', then TTY priority, then SAB priority,
        # then shorter string, then lexicographic stability
        ispref = 1 if str(r.get("ISPREF", "")).upper() == "Y" else 0
        ts_pref = 1 if str(r.get("TS", "")).upper() == "P" else 0
        tty = str(r.get("TTY", ""))
        sab = str(r.get("SAB", ""))
        strval = str(r.get("STR", ""))

        return (
            ispref,
            ts_pref,
            tty_rank.get(tty, 0),
            sab_rank.get(sab, 0),
            -len(strval),   # shorter is better
            strval.lower(), # stable tie-break
        )

    # 4) pick best row
    best = max(df.to_dict("records"), key=score_row)
    return best.get("STR")


In [191]:
mrconso_df = pd.read_csv('/mnt/share/kaichixie/A2H_preclinical_database/data/UMLS/2024AB/META/MRCONSO.RRF',
                         sep='|', header=None, dtype=str).iloc[:, :-1]
mrconso_df.columns = [
    'CUI', 'LAT', 'TS', 'LUI', 'STT', 'SUI', 'ISPREF', 'AUI', 'SAUI',
    'SCUI', 'SDUI', 'SAB', 'TTY', 'CODE', 'STR', 'SRL', 'SUPPRESS', 'CVF'
]

In [197]:
from pandarallel import pandarallel

pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [198]:
pcdb_drug_entities['drug_name'] = \
    pcdb_drug_entities['drug_name'].parallel_apply(lambda x: representative_name_for_cui(mrconso_df, x[5:]) if x.startswith("UMLS") else x)

In [202]:
pcdb_drug_entities.columns

Index(['drugbank_id', 'PCDB_id', 'mesh_id', 'UMLS_id', 'drug_name',
       'drug_synonyms'],
      dtype='object')

In [203]:
pcdb_drug_entities = pcdb_drug_entities[[
    'PCDB_id', 'drug_name', 'drug_synonyms', 
    'drugbank_id', 'mesh_id', 'UMLS_id'
]]

In [204]:
pcdb_drug_entities.to_csv(os.path.join('/mnt/share/kaichixie/A2H_preclinical_database/outputs/preclinical_db/v3',
                                          "drug_entities.tsv"),
                                          sep='\t',
                                          index=False)

In [165]:
pcdb_id_lut_only_db_ids = {}
first_cover_drugbank = pcdb_drug_entities[pcdb_drug_entities['drugbank_id']!='']
for pcdb_id, db_id in zip(first_cover_drugbank["PCDB_id"].tolist(),
                            first_cover_drugbank["drugbank_id"].tolist()):
    pcdb_id_lut_only_db_ids[db_id] = pcdb_id

In [166]:
second_cover_mesh = \
    pcdb_drug_entities[pcdb_drug_entities['mesh_id'].apply(lambda x: (x!='') and (isinstance(x, str)))]
pcdb_id_lut_only_mesh_ids = {}
for pcdb_id, mesh_id in zip(second_cover_mesh["PCDB_id"].tolist(),
                            second_cover_mesh["mesh_id"].tolist()):
    pcdb_id_lut_only_mesh_ids[mesh_id] = pcdb_id

In [167]:
third_cover_umls = \
    pcdb_drug_entities[pcdb_drug_entities['UMLS_id'].apply(lambda x: (x!='') and (isinstance(x, str)))]
pcdb_id_lut_only_umls_ids = {}
for pcdb_id, umls_id in zip(third_cover_umls["PCDB_id"].tolist(),
                            third_cover_umls["UMLS_id"].tolist()):
    pcdb_id_lut_only_umls_ids[umls_id] = pcdb_id

In [177]:
def get_pcdb_id(row):
    d_eids = row["drug_external_ids"]
    drug_names = row["drug_names"]
    unresolved_drug_names = []
    unresolved_drug_external_ids = []
    pcdb_ids = []
    for drug_name, drug_eid in zip(drug_names, d_eids):
        resolved = False
        if 'drugbank' in drug_eid:
            for db_id in drug_eid["drugbank"]:
                pcdb_ids.append(pcdb_id_lut_only_db_ids[db_id])
                resolved = True
        elif 'mesh' in drug_eid:
            for mesh_id in drug_eid["mesh"]:
                if mesh_id in pcdb_id_lut_only_mesh_ids:
                    pcdb_ids.append(pcdb_id_lut_only_mesh_ids[mesh_id])
                    resolved = True
        elif 'UMLS' in drug_eid:
            for umls_id in drug_eid["UMLS"]:
                if umls_id in pcdb_id_lut_only_umls_ids:
                    pcdb_ids.append(pcdb_id_lut_only_umls_ids[umls_id])
                    resolved = True
        if not resolved:
            unresolved_drug_names.append(drug_name)
            unresolved_drug_external_ids.append(drug_eid)
            
    if len(pcdb_ids) == 0:
        pcdb_ids = ''
    if len(unresolved_drug_names) == 0:
        unresolved_drug_names = ''
        unresolved_drug_external_ids = ''

    return pcdb_ids, unresolved_drug_names, unresolved_drug_external_ids
        
pcdb[["drug_pcdb_ids", 'unresolved_drug_names', 'unresolved_drug_external_ids']] = pcdb.apply(lambda x: get_pcdb_id(x), result_type='expand', axis=1)

In [174]:
pcdb['drug_names'] = pcdb['drug_names'].apply(lambda x: ast.literal_eval(x) if x != '' else '')

In [169]:
pcdb[pcdb['drug_pcdb_ids'] != '']['pmcid'].nunique()

79796

In [178]:
pcdb.head(50)

,pmcid,date,link,confidence_score,TITLE,disease_names,disease_external_ids,drug_names,drug_external_ids,animal_strain,animal_species,total_subject_size,animal_external_ids,drug_pcdb_ids,unresolved_drug_names,unresolved_drug_external_ids
0,PMC10000236,2023-02-22,https://pmc.ncbi.nlm.nih.gov/articles/PMC10000236,0.9972,The Anti-Tumor Effect of the Newly Developed L...,['colorectal cancer'],"[{'mesh': ['D015179'], 'diseaseontology': ['DO...",[jph203],"[{'pubchem': ['CID:24853505'], 'mesh': ['C5481...",balb/c,mouse,,{'mesh': 'D051379'},,[jph203],"[{'pubchem': ['CID:24853505'], 'mesh': ['C5481..."
1,PMC10000284,2021-10-18,https://pmc.ncbi.nlm.nih.gov/articles/PMC10000284,0.9948,KDM1A inhibition augments the efficacy of rapa...,['endometrial cancer'],"[{'mesh': ['D016889'], 'diseaseontology': ['DO...","[ncd38, rapamycin, sp2509]","[{'pubchem': ['CID:127031303']}, {'pubchem': [...","athymic, nude",mouse,16,{'mesh': 'D051379'},[PCDB_DR1916],"[ncd38, sp2509]","[{'pubchem': ['CID:127031303']}, {'pubchem': [..."
2,PMC10000284,2021-10-18,https://pmc.ncbi.nlm.nih.gov/articles/PMC10000284,0.9948,KDM1A inhibition augments the efficacy of rapa...,['endometrial cancer'],"[{'mesh': ['D016889'], 'diseaseontology': ['DO...","[ncd38, rapamycin, sp2509]","[{'pubchem': ['CID:127031303']}, {'pubchem': [...",nsg,mouse,14-16,{'mesh': 'D051379'},[PCDB_DR1916],"[ncd38, sp2509]","[{'pubchem': ['CID:127031303']}, {'pubchem': [..."
3,PMC10000367,2023-03-02,https://pmc.ncbi.nlm.nih.gov/articles/PMC10000367,0.9399,"Micrandilactone C, a Nortriterpenoid Isolated ...","[""huntington's disease""]","[{'mesh': ['D006816'], 'diseaseontology': ['DO...",[methylcobalamin],"[{'pubchem': ['CID:10898559', 'CID:71306319', ...",c57bl/6,mouse,105,{'mesh': 'D051379'},[PCDB_DR4424],,
4,PMC10000396,2023-03-06,https://pmc.ncbi.nlm.nih.gov/articles/PMC10000396,0.9876,Sodium New Houttuyfonate Induces Apoptosis of ...,['breast cancer'],"[{'mesh': ['D001943'], 'diseaseontology': ['DO...","[sodium new houttuyfonate, docetaxel]","[{'pubchem': ['CID:44602651'], 'mesh': ['C5563...",balb/c,mouse,,{'mesh': 'D051379'},[PCDB_DR1180],[sodium new houttuyfonate],"[{'pubchem': ['CID:44602651'], 'mesh': ['C5563..."
5,PMC10000467,2023-03-03,https://pmc.ncbi.nlm.nih.gov/articles/PMC10000467,0.9325,Melatonin Inhibits VEGF-Induced Endothelial Pr...,['neovascular age-related macular degeneration'],"[{'diseaseontology': ['DOID:10873'], 'UMLS': [...","[bevacizumab, melatonin]","[{'drugbank': ['DB00112'], 'mesh': ['D00006825...",c57bl/6j,mouse,,{'mesh': 'D051379'},"[PCDB_DR167, PCDB_DR4371]",,
6,PMC10000467,2023-03-03,https://pmc.ncbi.nlm.nih.gov/articles/PMC10000467,0.9325,Melatonin Inhibits VEGF-Induced Endothelial Pr...,['neovascular age-related macular degeneration'],"[{'diseaseontology': ['DOID:10873'], 'UMLS': [...","[bevacizumab, melatonin]","[{'drugbank': ['DB00112'], 'mesh': ['D00006825...",,chicken,,{'mesh': 'D002645'},"[PCDB_DR167, PCDB_DR4371]",,
7,PMC10000472,2023-02-22,https://pmc.ncbi.nlm.nih.gov/articles/PMC10000472,0.9141,Sulforaphane Potentiates Gemcitabine-Mediated ...,['intrahepatic cholangiocarcinoma'],"[{'mesh': ['D018281'], 'diseaseontology': ['DO...","[sulforaphane, gemcitabine]","[{'pubchem': ['CID:5350'], 'drugbank': ['DB124...",balb/cslc-nu/nu,mouse,80,{'mesh': 'D051379'},"[PCDB_DR309, PCDB_DR3059]",,
8,PMC10000671,2023-03-04,https://pmc.ncbi.nlm.nih.gov/articles/PMC10000671,0.9903,Evaluation of Electrochemotherapy with Bleomyc...,"['hepatic cancer', 'colorectal hepatic metasta...","[{'mesh': ['D008113'], 'diseaseontology': ['DO...","[electrochemotherapy, bleomycin]","[{'mesh': ['D053672'], 'UMLS': ['C4721571']}, ...",wag/rij,rat,32,{'mesh': 'D051381'},[PCDB_DR2990],[electrochemotherapy],"[{'mesh': ['D053672'], 'UMLS': ['C4721571']}]"
9,PMC10000780,2023-02-23,https://pmc.ncbi.nlm.nih.gov/articles/PMC10000780,0.9770,Stem Cell Factor and Granulocyte Colony-Stimul...,['traumatic brain injury'],"[{'mesh': ['D000070642'], 'diseaseontology': [...","[stem cell factor, granulocyte col

In [107]:
pcdb[pcdb['drug_external_ids'] != '']['pmcid'].nunique()

104381

In [110]:
79796 / 104381

0.764468629348253

In [ ]:
# get descriptor name, syonyms (all DB, MeSH, UMLS only allowed categories)

In [ ]:
# umls_only_ids = \
#     pcdb_drug_entities[(pcdb_drug_entities['drugbank_id'] == '') & (pcdb_drug_entities['mesh_id'] == '')]['UMLS_id'].to_list()
# prop_umls_ids = \
#     pcdb_drug_entities[(pcdb_drug_entities['drugbank_id'] != '') | (pcdb_drug_entities['mesh_id'] != '')]['UMLS_id'].to_list()
# prop_umls_ids = list(chain.from_iterable(prop_umls_ids))

# for k in umls_only_ids:
#     if k in prop_umls_ids:
#         print(k)